In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Estimativa de uso: um minuto em um processador Eagle (NOTA: Isso é apenas uma estimativa. Seu tempo de execução pode variar.)*

## Contexto

Circuit-knitting é um termo abrangente que engloba vários métodos de particionamento de um circuito em múltiplos subcircuitos menores envolvendo menos portas e/ou qubits. Cada um dos subcircuitos pode ser executado independentemente e o resultado final é obtido por algum pós-processamento clássico sobre o resultado de cada subcircuito. Esta técnica está acessível no [addon Qiskit de corte de circuito](https://qiskit.github.io/qiskit-addon-cutting/index.html), uma explicação detalhada da técnica é fornecida na [documentação](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html) junto com outro [material introdutório](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html).

Este notebook trata de um método chamado <b>corte de fios</b> onde o circuito é particionado ao longo do fio [\[1\], \[2\]](#references). Observe que o particionamento é simples em circuitos clássicos, pois o resultado no ponto de partição pode ser determinado deterministicamente, e é 0 ou 1. No entanto, o estado do qubit no ponto do corte é, em geral, um estado misto. Portanto, cada subcircuito precisa ser medido múltiplas vezes em diferentes bases (geralmente um conjunto tomograficamente completo de bases como a base de Pauli [\[3\], \[4\]](#references) e correspondentemente preparado em seu autoestado. A figura abaixo (<i>cortesia: Tese de PhD, Ritajit Majumdar</i>) mostra um exemplo de corte de fios para um estado GHZ de 4 qubits em três subcircuitos. Aqui $M_j$ denota um conjunto de bases (geralmente Pauli X, Y e Z) e $P_i$ denota um conjunto de autoestados (geralmente $|0\rangle$, $|1\rangle$, $|+\rangle$ e $|+i\rangle$).

![wc-1.png](../docs/images/tutorials/wire-cutting-to-improve-performance/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting-to-improve-performance/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

Como cada subcircuito tem menos qubits e/ou portas, espera-se que eles sejam menos suscetíveis ao ruído. Este notebook mostra um exemplo onde este método pode ser usado para suprimir efetivamente o ruído no sistema.

## Requisitos

Antes de iniciar este tutorial, certifique-se de ter o seguinte instalado:

- Qiskit SDK v2.0 ou posterior, com suporte de [visualização](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 ou posterior ( `pip install qiskit-ibm-runtime` )
- Addon Qiskit de corte de circuito v0.9.0 ou posterior (`pip install qiskit-addon-cutting`)

Vamos considerar um circuito de Localização de Muitos Corpos (MBL) para este notebook. O circuito MBL é um circuito eficiente em hardware e é parametrizado por dois parâmetros $\theta$ e $\vec{\phi}$. Quando $\theta$ é definido como $0$ e o estado inicial é preparado em $|0\rangle$ para todos os qubits, o valor esperado ideal de $\langle Z_i \rangle$ é $+1$ para cada site de qubit $i$ independentemente dos valores de $\vec{\phi}$. Você pode verificar mais detalhes sobre circuitos MBL em <a href="https://arxiv.org/abs/2307.07552">este artigo</a>.

## Configuração

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## Parte I. Exemplo em pequena escala

### Passo 1: Mapear entradas clássicas para um problema quântico

Inicialmente construímos um circuito modelo sem nenhum valor de parâmetro específico. Também fornecemos espaços reservados, chamados `CutWire`, para anotar a posição dos cortes. Para o exemplo em pequena escala, consideramos um circuito MBL de 10 qubits.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

We calculate the average expectation value $O = \frac{1}{n} \sum_i Z_i$ over all qubits for $\theta = 0$. Since the ideal expectation value of $\langle Z_i \rangle = 1$ $\forall$ $i$, the ideal expectation value of $O$ is also $1$. The parameters $\phi$ are selected randomly.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

Agora anotamos o circuito para corte inserindo o **CutWire** adequado para criar dois cortes aproximadamente iguais. Definimos `use_cut=True` na função e permitimos que ela anote após $\frac{n}{2}$ qubits, sendo $n$ o número de qubits no circuito original.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/31844134-514b-46ea-85f9-133e432f053f-0.avif)

### Passo 2: Otimizar problema para execução em hardware quântico
Em seguida, cortamos o circuito em dois subcircuitos menores. Para este exemplo, nos limitamos a apenas 2 subcircuitos. Para isso, usamos o <a href="https://qiskit.github.io/qiskit-addon-cutting/">Qiskit Addon: Circuit Cutting</a>.
#### Cortar o circuito em subcircuitos menores
Cortar o fio em um ponto aumenta a contagem de qubits em um. Além do qubit original, agora existe um qubit extra como espaço reservado para o circuito após o corte. A imagem a seguir fornece uma representação:

![wc-4.png](../docs/images/tutorials/wire-cutting-to-improve-performance/dfc5f923-e507-4873-888e-d90e1618be3a.avif)

Este Addon usa a função `cut_wires` para contabilizar os qubits extras decorrentes do corte.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

#### Criar e expandir os observáveis
Agora construímos o observável $M_z = \frac{1}{n}\sum_{i=1}^n \langle Z_i \rangle$. Como o resultado ideal de $\langle Z_i \rangle$ para cada $i$ é $+1$, o resultado ideal de $M_z$ também é $+1$.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

No entanto, observe que o número de qubits no circuito aumentou após inserir as operações virtuais `Move` de 2 qubits após o corte. Portanto, precisamos expandir os observáveis também inserindo identidades para afirmar ao circuito atual.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

Vamos visualizar os subcircuitos

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

#### Transpile the circuits onto the backend

For the first example involving only simulation, we transpile the circuit into the basis gate set of the backend:

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/35920640-76e8-4af6-a252-ee6a22e9c26a-0.avif)

Os observáveis também foram particionados para se ajustar aos subcircuitos

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

Observe que cada subcircuito leva a um número de amostras. A reconstrução leva em conta o resultado de cada uma dessas amostras. Cada uma dessas amostras é denominada `subexperiment`.
Estender o observável usando a operação `Move` requer uma estrutura de dados `PauliList`. Também podemos criar o observável $M_z$ na estrutura de dados `SparsePauliOp` mais genérica, que será útil posteriormente durante a reconstrução dos subexperimentos.

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

Vejamos dois exemplos onde os qubits cortados são medidos em duas bases diferentes. Primeiro, é medido na base Z normal, e depois é medido na base X.

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/987547e4-296a-41e4-ad82-41f4139a87a0-0.avif)

#### Transpilar cada subexperimento

Atualmente precisamos transpilar nossos circuitos antes de submetê-los para execução. Portanto, vamos transpilar cada circuito nos subexperimentos primeiro.